In [ ]:
import pandas as pd
df_customer = pd.read_csv(r"C:\Users\meski\OneDrive\Desktop\python-project\customer_data.csv")
# data quality check 
df_customer.info()
df_customer.describe(include="object")
# data cleaning 
# duplicate handling 
print(df_customer.duplicated().sum())
# missing value check
df_customer.isna().sum()
# column email has 496 missing values in the data set so we prefer to substitute it by unknown
df_customer["email"] = df_customer["email"].fillna("unknown")
df_customer

#Fix column names
df_customer.columns = (
    df_customer.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")

)
df_customer
# Outlier Detection on Age
df_customer["age"].describe()
Q1 = df_customer["age"].quantile(0.25)
Q3 = df_customer["age"].quantile(0.75)

IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

Q1, Q3, IQR, lower_bound, upper_bound
age_outliers = df_customer[
    (df_customer["age"] < lower_bound) |
     (df_customer["age"] > upper_bound)
]

age_outliers
# we don't have age outliers in the customer table but still we can set an invalid limit
# Remove invalid values (validity is taken for age <= 100)
df = df_customer[(df_customer["age"] >= 0) & (df_customer["age"] <= 100)]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  25000 non-null  object
 1   age          25000 non-null  int64 
 2   gender       25000 non-null  object
 3   city         25000 non-null  object
 4   email        24504 non-null  object
dtypes: int64(1), object(4)
memory usage: 976.7+ KB
0


,customer_id,age,gender,city,email


In [ ]:
import pandas as pd
df_sales = pd.read_csv(r"C:\Users\meski\OneDrive\Desktop\python-project\sales_data.csv")
# data quality check 
df_sales.info()
df_sales.describe()
# data cleaning 
# duplicate handling 
print(df_sales.duplicated().sum())
# fix column names 
df_sales.columns = (
     df_sales.columns
     .str.strip()
     .str.lower()
     .str.replace(" ", "_")

 )
df_sales
# date parsing 
# date Dtype is object so we need to change in to datetime
import pandas as pd
df_sales["date"] = pd.to_datetime(
df_sales["date"],
errors="coerce"
 )
# check the dtype
print(df_sales["date"].dtype)
print(df_sales["date"].head())
#  invalid dates will be NaT
df_sales["date"].isna().sum()

# returned Dtype is int64 ,should be changed to boolean
df_sales["returned"] = df_sales["returned"].astype(bool)

# check the dtype and amounts
df_sales["returned"].dtype
df_sales["returned"].value_counts()
# missing value check 
df_customer.isna().sum()
#customer_id and discount has missing values 
#customer_id has missing values and prefered to fill it with unknown
df_sales["customer_id"] = df_sales["customer_id"].fillna("unknown")
df_sales
# discount has missing value and prefered to fill it with 0
df_sales["discount"] = df_sales["discount"].fillna(0)
 

 



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  50000 non-null  object 
 1   date            50000 non-null  object 
 2   product_id      50000 non-null  object 
 3   store_id        50000 non-null  object 
 4   customer_id     48156 non-null  object 
 5   quantity        50000 non-null  int64  
 6   discount        47417 non-null  float64
 7   returned        50000 non-null  int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 3.1+ MB
datetime64[ns]
0   2023-08-02
1   2020-08-16
2   2020-02-21
3   2024-02-27
4   2021-07-17
Name: date, dtype: datetime64[ns]


customer_id    0
age            0
gender         0
city           0
email          0
dtype: int64

In [36]:
import pandas as pd
df_product = pd.read_csv(r"C:\Users\meski\OneDrive\Desktop\python-project\product_data.csv")
# data quality check 
df_product.info()
#data cleaning 
# duplicate handling 
print(df_product.duplicated().sum())
# fix column names 
df_product.columns = (
    df_product.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")

 )
df_product
#missing value check
df_product.isna().sum()
# column color has 990 missing value and check column category has an invalid value???
df_product["category"].value_counts()
#499 ??? invalide values obtained in category column
# missing value will be drop and the invalid value will be replace with unkown 
df_product["color"] = df_product["color"].dropna
df_product["category"] = df_product["category"].replace("???" , "unknown")
# check for invalide prices
df_product["cost_price"].describe()
df_product["list_price"].describe() 
# invalide price removal
df_product = df_product[
    (df_product["cost_price"] > 0) &
    (df_product["list_price"] > 0)
]

# Aggregate revenue by month & customer
# to get revenue join df_sales with df_product
df = df_sales.merge(
    df_product[["product_id", "list_price"]],
    on="product_id",
    how="left"
)
# revenue 
df["revenue"] = df["quantity"] * df["list_price"] * (1 - df["discount"])
df["revenue"]
# we need to extract month since we are asked monthly aggregation
df["year_month"] = df["date"].dt.to_period("M")
df["year_month"]
# revenue by month & customer
monthly_customer_revenue = (
    df.groupby(["year_month", "customer_id"], as_index=False)
      .agg(
          total_revenue=("revenue", "sum"),
          total_transactions=("transaction_id", "count"),
          total_items=("quantity", "sum")
      )
)
monthly_customer_revenue





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  50000 non-null  object 
 1   category    50000 non-null  object 
 2   color       49010 non-null  object 
 3   size        50000 non-null  object 
 4   season      50000 non-null  object 
 5   supplier    50000 non-null  object 
 6   cost_price  50000 non-null  float64
 7   list_price  50000 non-null  float64
dtypes: float64(2), object(6)
memory usage: 3.1+ MB
0


,year_month,customer_id,total_revenue,total_transactions,total_items
0,2020-01,C000009,262.188,1,2
1,2020-01,C000027,598.500,1,4
2,2020-01,C000031,572.480,1,4
3,2020-01,C000053,516.600,1,4
4,2020-01,C000085,492.990,1,3
...,...,...,...,...,...
47380,2024-12,C024856,381.000,1,4
47381,2024-12,C024870,196.730,1,1
47382,2024-12,C024957,91.060,1,2
47383,2024-12,C024976,173.270,1,1


In [38]:
df_store = pd.read_csv(r"C:\Users\meski\OneDrive\Desktop\python-project\store_data.csv")
df_store
# data quality check 
df_store.info()
#data cleaning 
# duplicate handling 
print(df_store.duplicated().sum())
# fix column names 
df_store.columns = (
    df_store.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")

 )
df_store
#missing value check
df_store.isna().sum()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   store_id       5 non-null      object
 1   store_name     5 non-null      object
 2   region         5 non-null      object
 3   store_size_m2  5 non-null      int64 
dtypes: int64(1), object(3)
memory usage: 292.0+ bytes
0


store_id         0
store_name       0
region           0
store_size_m2    0
dtype: int64